# Установка окружения и визуализация в Google Colab

В этом ноутбуке показано, как в Google Colab подключить Google Drive, скачать репозиторий курса, загрузить метеоданные и построить графики временных рядов.


## Монтируем гугл-диск к виртуальной машине

Подключаем Google Drive к виртуальной машине Colab, чтобы читать и сохранять файлы на диске.

**Инструменты:** [google.colab.drive.mount](https://colab.research.google.com/notebooks/io.ipynb).

In [ ]:
import sys, os
from google.colab import drive
drive.mount('/content/drive')

## Скачиваем репозиторий курса

Клонируем (или обновляем) репозиторий курса на Google Drive.

**Инструменты:** команды [git](https://git-scm.com/doc) и [магические команды IPython](https://ipython.readthedocs.io/en/stable/interactive/magics.html) (`%cd`, `!git clone`).

In [ ]:


%cd /content/drive/MyDrive/
%pwd


os.makedirs ('UClim4HSE', exist_ok = True)

%cd /content/drive/MyDrive/UClim4HSE/
%pwd

if not os.path.isdir('/content/drive/MyDrive/UClim4HSE/UClim4HSE-2025'):
  print ('cloning repository')
  !git clone https://github.com/mvarentsov/UClim4HSE-2025.git
else:
  print ('updating repository')
  %cd /content/drive/MyDrive/UClim4HSE/UClim4HSE-2025/
  ! git pull

%cd /content/drive/MyDrive/UClim4HSE/


In [ ]:
sys.path.append('/content/drive/MyDrive/UClim4HSE/UClim4HSE-2025/scripts/')
import meteo_utils
from datetime import date, datetime, timedelta

In [ ]:

station_id = 27605     # идентификатор станции, с которой нужны наблюдения
start = date (2023,1,1) # начало периода загрузки данных
end = date (2024,1,1) #date.today()      # конец  периода загрузки данных

meteo_utils.load_weaclim4period (station_id, start, end, 'weaclim_data', adress="http://www.pogodaiklimat.ru/weather.php?")


In [ ]:
df_urb = meteo_utils.read_weaclim_dir ('weaclim_data/27605')
df_rur = meteo_utils.read_weaclim_dir ('weaclim_data/27518')

df_rur.head()

In [ ]:
df_urb = df_urb.rename(columns = {x:'urb_' + x for x in df_urb.columns})
df_rur = df_rur.rename(columns = {x:'rur_' + x for x in df_rur.columns})

df = df_urb.merge (df_rur, left_index=True, right_index=True)
display(df.head())


In [ ]:
from matplotlib import pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 200

# plt.figure()
# df['rur_t2m'].plot(color = 'blue')
# df['urb_t2m'].plot(color = 'red')
# plt.ylabel ('Temperature[K]')

# df_sel = df['2023-08':'2023-09']
# plt.figure()
# df_sel['rur_t2m'].plot(color = 'blue')
# df_sel['urb_t2m'].plot(color = 'red')
# plt.ylabel ('Temperature[K]')

#df_sel = df['2023-08':'2023-09-01']
plt.figure()
df['rur_t2m'].plot(color = 'blue', label = 'rural')
df['urb_t2m'].plot(color = 'red', label = 'urban')
plt.ylabel ('Temperature [K]')
plt.xlabel ('Date')
plt.title ('Surface air temperature, August 2023')
plt.legend()
plt.xlim((datetime(2023,8,1), datetime(2023,9,1)))



## Интерактивный график с помощью plotly

Строим интерактивные графики временных рядов температуры с помощью Plotly Express.

**Инструменты:** [plotly.express.line](https://plotly.com/python/line-charts/), [pandas.Series.plot](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.html) и [matplotlib.pyplot](https://matplotlib.org/stable/api/pyplot_summary.html).

In [ ]:

import plotly.express as px

fig = px.line(df, y=['rur_t2m', 'urb_t2m'], title='Surface air temperature (urban vs rural)')
fig.update_yaxes (title_text = 'Temperature [K]')
fig.update_xaxes (title_text = 'Date')
fig.data[0].line.color = 'blue'  # Color for 'rur_t2m'
fig.data[1].line.color = 'orange'  # Color for 'urb_t2m'
fig.show()

In [ ]:
#df_day_mean = df.resample ('1ME').mean()
df_day_mean = df.groupby(df.index.month).min()


display(df_day_mean)

fig = px.line(df_day_mean, y=['rur_t2m', 'urb_t2m'], title='Monthly minimum surface air temperature')
fig.update_yaxes (title_text = 'Temperature [K]')
fig.update_xaxes (title_text = 'Month')
fig.data[0].line.color = 'blue'  # Color for 'rur_t2m'
fig.data[1].line.color = 'orange'  # Color for 'urb_t2m'
fig.show()
